# 05. 전월세 EDA & Feature Engineering

서울 아파트 매매가격 예측에 사용할 **과거 전세시장 피처**를 생성한다.

핵심 원칙:
- 전세시장 정보는 매매 거래월보다 최소 1개월 이전 정보만 사용
- 단지 → 법정동 → 자치구 순으로 전세가격/거래량 fallback
- 현재 매매가격에서 파생된 값은 전월세 피처에 사용하지 않음
- 최종 결과는 `transaction_id` 기준 **feature-only parquet**으로 저장

최종 산출물:
- `df_fe_rent.parquet`

## 1. 설정 및 데이터 로드

전월세 원본과 매매 기본 피처만 사용한다.
학교·지하철·생활입지 피처와 동일하게 전월세도 별도 feature-only 파일로 관리한다.

In [ ]:
# ============================================================
# 라이브러리 및 기본 설정
# ============================================================

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
# ============================================================
# 경로 및 분석 설정
# ============================================================

def find_project_dir():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. "
        "data 폴더가 있는 위치에서 노트북을 실행해 주세요."
    )


PROJECT_DIR = find_project_dir()

DATA_DIR = PROJECT_DIR / "data" / "v2_5year_seoul"
PROCESSED_DIR = DATA_DIR / "processed"

RENT_RAW_PATH = PROCESSED_DIR / "apt_rent_raw.parquet"
TRADE_BASE_PATH = PROCESSED_DIR / "df_fe_base.parquet"
RENT_FEATURE_PATH = PROCESSED_DIR / "df_fe_rent.parquet"

MODEL_START = pd.Timestamp("2021-07-01")
MODEL_END = pd.Timestamp("2026-06-01")

ROLLING_WINDOWS = (3, 6)
PRICE_MIN_MONTHS = {
    3: 2,
    6: 3,
}
APT_MIN_TRADES = 3

print("PROJECT_DIR:", PROJECT_DIR)
print("전월세:", RENT_RAW_PATH)
print("매매 기본 피처:", TRADE_BASE_PATH)


In [ ]:
# ============================================================
# 입력 데이터 로드
# ============================================================

required_paths = [
    RENT_RAW_PATH,
    TRADE_BASE_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        f"필수 입력 파일이 없습니다: {missing_paths}"
    )

rent_raw = pd.read_parquet(RENT_RAW_PATH)
trade_base = pd.read_parquet(TRADE_BASE_PATH)

print("전월세 원본:", rent_raw.shape)
print("매매 기본 피처:", trade_base.shape)


## 2. 전월세 원본 검증 및 전처리

전세/월세 구분, 날짜, 금액, 면적을 정리한다.

공개 전월세 데이터는 계약을 유일하게 구분하는 행 번호가 없으므로
조건이 동일한 거래를 임의로 중복 삭제하지 않는다.

In [ ]:
# ============================================================
# Raw Data 검증
# ============================================================

required_raw_cols = [
    "aptSeq",
    "aptNm",
    "sggCd",
    "umdNm",
    "dealYear",
    "dealMonth",
    "dealDay",
    "deposit",
    "monthlyRent",
    "excluUseAr",
    "floor",
    "buildYear",
    "수집_자치구",
    "수집_시군구코드",
]

missing_raw_cols = [
    col
    for col in required_raw_cols
    if col not in rent_raw.columns
]

if missing_raw_cols:
    raise KeyError(
        f"전월세 원본 필수 컬럼이 없습니다: {missing_raw_cols}"
    )

raw_month = pd.to_datetime(
    {
        "year": pd.to_numeric(
            rent_raw["dealYear"],
            errors="coerce",
        ),
        "month": pd.to_numeric(
            rent_raw["dealMonth"],
            errors="coerce",
        ),
        "day": 1,
    },
    errors="coerce",
)

raw_summary = pd.Series(
    {
        "행 수": len(rent_raw),
        "컬럼 수": len(rent_raw.columns),
        "최초 거래월": raw_month.min(),
        "최종 거래월": raw_month.max(),
        "거래월 수": raw_month.dt.to_period("M").nunique(),
        "자치구 수": rent_raw["수집_시군구코드"].nunique(),
        "완전 동일 행": rent_raw.duplicated().sum(),
    },
    name="값",
)

display(raw_summary.to_frame())


In [ ]:
# ============================================================
# 기본 전처리
# ============================================================

rent = rent_raw.copy()
rent = rent.replace(
    ["nan", "None", "null", ""],
    pd.NA,
)

numeric_cols = [
    "dealYear",
    "dealMonth",
    "dealDay",
    "deposit",
    "monthlyRent",
    "excluUseAr",
    "floor",
    "buildYear",
]

for col in numeric_cols:
    rent[col] = pd.to_numeric(
        rent[col]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce",
    )

text_cols = [
    "aptNm",
    "umdNm",
    "수집_자치구",
]

for col in text_cols:
    rent[col] = (
        rent[col]
        .astype("string")
        .str.strip()
    )

rent["sggCd"] = (
    rent["sggCd"]
    .astype("string")
    .str.split(".")
    .str[0]
    .str.zfill(5)
)

rent["수집_시군구코드"] = (
    rent["수집_시군구코드"]
    .astype("string")
    .str.split(".")
    .str[0]
    .str.zfill(5)
)

rent["aptSeq"] = (
    rent["aptSeq"]
    .astype("string")
    .str.strip()
)

rent["거래일자"] = pd.to_datetime(
    {
        "year": rent["dealYear"],
        "month": rent["dealMonth"],
        "day": rent["dealDay"],
    },
    errors="coerce",
)

rent["거래월일자"] = (
    rent["거래일자"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

rent["연식"] = (
    rent["dealYear"]
    - rent["buildYear"]
)

invalid_age = (
    rent["연식"].lt(0)
    | rent["연식"].gt(100)
)

rent.loc[
    invalid_age,
    "연식",
] = np.nan

print("거래일자 결측:", rent["거래일자"].isna().sum())
print("비정상 연식:", invalid_age.sum())
print(
    "시군구코드 불일치:",
    (rent["sggCd"] != rent["수집_시군구코드"]).sum(),
)


In [ ]:
# ============================================================
# 전세·월세 구분 및 전세 평당가 생성
# ============================================================

rent["계약유형"] = pd.Series(
    pd.NA,
    index=rent.index,
    dtype="string",
)

rent.loc[
    rent["monthlyRent"].eq(0),
    "계약유형",
] = "전세"

rent.loc[
    rent["monthlyRent"].gt(0),
    "계약유형",
] = "월세"

valid_jeonse = (
    rent["계약유형"].eq("전세")
    & rent["deposit"].gt(0)
    & rent["excluUseAr"].gt(0)
    & rent["거래월일자"].notna()
)

jeonse = rent.loc[
    valid_jeonse
].copy()

jeonse["전세평당가"] = (
    jeonse["deposit"]
    / (jeonse["excluUseAr"] / 3.3058)
)

print("계약유형:")
print(rent["계약유형"].value_counts(dropna=False))

print("\n분석 가능 전세:", len(jeonse))
print(jeonse["전세평당가"].describe())


## 3. 전월세 EDA

극단값은 자동 제거하지 않고 확인만 한다.
이후 월별 집계에서는 중위값을 사용해 극단값 영향을 줄인다.

In [ ]:
# ============================================================
# 전월세 품질 및 시장 분포 확인
# ============================================================

quality_check = pd.Series(
    {
        "보증금_0": rent["deposit"].eq(0).sum(),
        "월세_음수": rent["monthlyRent"].lt(0).sum(),
        "면적_0이하": rent["excluUseAr"].le(0).sum(),
        "건축년도_거래년도초과": (
            rent["buildYear"] > rent["dealYear"]
        ).sum(),
    },
    name="건수",
)

monthly_eda = (
    rent.groupby("거래월일자")
    .agg(
        전체거래량=("계약유형", "size"),
        전세거래량=(
            "계약유형",
            lambda s: s.eq("전세").sum(),
        ),
        월세거래량=(
            "계약유형",
            lambda s: s.eq("월세").sum(),
        ),
    )
    .join(
        jeonse.groupby(
            "거래월일자"
        )["전세평당가"]
        .median()
        .rename("전세중위평당가")
    )
    .sort_index()
)

gu_eda = (
    jeonse.groupby(
        ["sggCd", "수집_자치구"]
    )
    .agg(
        전세거래량=("전세평당가", "size"),
        전세중위평당가=("전세평당가", "median"),
    )
    .sort_values(
        "전세중위평당가",
        ascending=False,
    )
)

display(quality_check.to_frame())
display(monthly_eda.tail(12))
display(gu_eda)


## 4. 과거 전세시장 Lag Feature

모든 가격·거래량 피처는 반드시 `shift(1)` 이후 계산한다.

- 전월: `t-1`
- 과거 3개월: `t-3 ~ t-1`
- 과거 6개월: `t-6 ~ t-1`

In [ ]:
# ============================================================
# Lag 생성 공통 함수
# ============================================================

ALL_MONTHS = pd.date_range(
    rent["거래월일자"].min(),
    rent["거래월일자"].max(),
    freq="MS",
)


def complete_month_panel(
    monthly,
    group_cols,
    count_cols,
):
    groups = (
        monthly[group_cols]
        .drop_duplicates()
    )

    months = pd.DataFrame(
        {"거래월일자": ALL_MONTHS}
    )

    panel = (
        groups
        .merge(months, how="cross")
        .merge(
            monthly,
            on=group_cols + ["거래월일자"],
            how="left",
            validate="one_to_one",
        )
    )

    for col in count_cols:
        panel[col] = (
            panel[col]
            .fillna(0)
            .astype("int64")
        )

    return (
        panel
        .sort_values(
            group_cols + ["거래월일자"]
        )
        .reset_index(drop=True)
    )


def add_lag_features(
    panel,
    group_cols,
    prefix,
    market_name,
    price_col,
    volume_col,
):
    result = (
        panel
        .sort_values(
            group_cols + ["거래월일자"]
        )
        .copy()
    )

    grouped = result.groupby(
        group_cols,
        dropna=False,
        group_keys=False,
    )

    result[
        f"{prefix}_전월_{market_name}평당가"
    ] = grouped[price_col].shift(1)

    result[
        f"{prefix}_전월_{market_name}거래량"
    ] = grouped[volume_col].shift(1)

    for window in ROLLING_WINDOWS:
        result[
            f"{prefix}_과거{window}개월_{market_name}평당가"
        ] = grouped[price_col].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    window=window,
                    min_periods=PRICE_MIN_MONTHS[window],
                )
                .median()
            )
        )

        result[
            f"{prefix}_과거{window}개월_{market_name}거래량"
        ] = grouped[volume_col].transform(
            lambda s: (
                s.shift(1)
                .rolling(
                    window=window,
                    min_periods=window,
                )
                .sum()
            )
        )

    return result


print(
    "월 범위:",
    ALL_MONTHS.min(),
    "~",
    ALL_MONTHS.max(),
)


In [ ]:
# ============================================================
# 자치구·법정동 전세 Lag Feature
# ============================================================

gu_monthly = (
    jeonse.groupby(
        ["sggCd", "거래월일자"]
    )
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)

gu_panel = complete_month_panel(
    gu_monthly,
    group_cols=["sggCd"],
    count_cols=["월전세거래량"],
)

gu_market_features = add_lag_features(
    gu_panel,
    group_cols=["sggCd"],
    prefix="자치구",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)


dong_monthly = (
    jeonse.groupby(
        [
            "sggCd",
            "umdNm",
            "거래월일자",
        ]
    )
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)

dong_panel = complete_month_panel(
    dong_monthly,
    group_cols=["sggCd", "umdNm"],
    count_cols=["월전세거래량"],
)

dong_market_features = add_lag_features(
    dong_panel,
    group_cols=["sggCd", "umdNm"],
    prefix="법정동",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)

print("자치구×월:", gu_market_features.shape)
print("법정동×월:", dong_market_features.shape)


In [ ]:
# ============================================================
# 단지 전세 Lag Feature
# ============================================================

apt_source = (
    jeonse.loc[
        jeonse["aptSeq"].notna()
    ]
    .copy()
)

apt_monthly = (
    apt_source.groupby(
        [
            "aptSeq",
            "sggCd",
            "umdNm",
            "거래월일자",
        ]
    )
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)

apt_panel = complete_month_panel(
    apt_monthly,
    group_cols=[
        "aptSeq",
        "sggCd",
        "umdNm",
    ],
    count_cols=["월전세거래량"],
)

apt_market_features = add_lag_features(
    apt_panel,
    group_cols=[
        "aptSeq",
        "sggCd",
        "umdNm",
    ],
    prefix="단지",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)

print(
    "전세 거래가 있는 단지:",
    apt_market_features["aptSeq"].nunique(),
)
print(
    "단지×월:",
    apt_market_features.shape,
)


## 5. 과거 매매시장과 전세가율

전세가율 역시 현재 거래의 매매가격을 사용하지 않는다.

법정동 단위의 과거 전세 평당가를 같은 기간의 **과거 매매 평당가 Lag**로 나눠 계산한다.

In [ ]:
# ============================================================
# 매매 기본 피처 검증 및 전처리
# ============================================================

required_trade_cols = [
    "transaction_id",
    "법정동시군구코드",
    "법정동",
    "단지일련번호",
    "거래월일자",
    "평당가",
]

missing_trade_cols = [
    col
    for col in required_trade_cols
    if col not in trade_base.columns
]

if missing_trade_cols:
    raise KeyError(
        f"매매 기본 피처 필수 컬럼이 없습니다: {missing_trade_cols}"
    )

trade = trade_base[
    required_trade_cols
].copy()

trade["법정동시군구코드"] = (
    trade["법정동시군구코드"]
    .astype("string")
    .str.split(".")
    .str[0]
    .str.zfill(5)
)

trade["법정동"] = (
    trade["법정동"]
    .astype("string")
    .str.strip()
)

trade["단지일련번호"] = (
    trade["단지일련번호"]
    .astype("string")
    .str.strip()
)

trade["거래월일자"] = pd.to_datetime(
    trade["거래월일자"]
)

print("매매 행 수:", len(trade))
print(
    "transaction_id 중복:",
    trade["transaction_id"].duplicated().sum(),
)


In [ ]:
# ============================================================
# 자치구·법정동 매매 Lag Feature 및 전세가율
# ============================================================

trade_gu_monthly = (
    trade.groupby(
        [
            "법정동시군구코드",
            "거래월일자",
        ]
    )
    .agg(
        월중위매매평당가=("평당가", "median"),
        월매매거래량=("평당가", "size"),
    )
    .reset_index()
    .rename(
        columns={
            "법정동시군구코드": "sggCd",
        }
    )
)

trade_gu_panel = complete_month_panel(
    trade_gu_monthly,
    group_cols=["sggCd"],
    count_cols=["월매매거래량"],
)

trade_gu_features = add_lag_features(
    trade_gu_panel,
    group_cols=["sggCd"],
    prefix="자치구",
    market_name="매매",
    price_col="월중위매매평당가",
    volume_col="월매매거래량",
)


trade_dong_monthly = (
    trade.groupby(
        [
            "법정동시군구코드",
            "법정동",
            "거래월일자",
        ]
    )
    .agg(
        월중위매매평당가=("평당가", "median"),
        월매매거래량=("평당가", "size"),
    )
    .reset_index()
    .rename(
        columns={
            "법정동시군구코드": "sggCd",
            "법정동": "umdNm",
        }
    )
)

trade_dong_panel = complete_month_panel(
    trade_dong_monthly,
    group_cols=["sggCd", "umdNm"],
    count_cols=["월매매거래량"],
)

trade_dong_features = add_lag_features(
    trade_dong_panel,
    group_cols=["sggCd", "umdNm"],
    prefix="법정동",
    market_name="매매",
    price_col="월중위매매평당가",
    volume_col="월매매거래량",
)

trade_gu_price_cols = [
    f"자치구_{label}_매매평당가"
    for label in [
        "전월",
        "과거3개월",
        "과거6개월",
    ]
]

trade_dong_price_cols = [
    f"법정동_{label}_매매평당가"
    for label in [
        "전월",
        "과거3개월",
        "과거6개월",
    ]
]

gu_market_features = gu_market_features.merge(
    trade_gu_features[
        [
            "sggCd",
            "거래월일자",
        ]
        + trade_gu_price_cols
    ],
    on=["sggCd", "거래월일자"],
    how="left",
    validate="one_to_one",
)

dong_market_features = dong_market_features.merge(
    trade_dong_features[
        [
            "sggCd",
            "umdNm",
            "거래월일자",
        ]
        + trade_dong_price_cols
    ],
    on=[
        "sggCd",
        "umdNm",
        "거래월일자",
    ],
    how="left",
    validate="one_to_one",
)

for label in [
    "전월",
    "과거3개월",
    "과거6개월",
]:
    dong_sale_price = (
        dong_market_features[
            f"법정동_{label}_매매평당가"
        ]
        .replace(0, np.nan)
    )

    gu_sale_price = (
        gu_market_features[
            f"자치구_{label}_매매평당가"
        ]
        .replace(0, np.nan)
    )

    dong_market_features[
        f"법정동_{label}_전세가율"
    ] = (
        dong_market_features[
            f"법정동_{label}_전세평당가"
        ]
        / dong_sale_price
    )

    gu_market_features[
        f"자치구_{label}_전세가율"
    ] = (
        gu_market_features[
            f"자치구_{label}_전세평당가"
        ]
        / gu_sale_price
    )

ratio_cols = [
    "법정동_전월_전세가율",
    "법정동_과거3개월_전세가율",
    "법정동_과거6개월_전세가율",
]

display(
    dong_market_features[
        ratio_cols
    ].describe().T
)


## 6. 매매 거래별 전월세 피처 생성

매매 거래월과 지역·단지 키를 이용해 Lag 테이블을 붙인다.

전세가격과 거래량은:
**단지 → 법정동 → 자치구** 순으로 fallback한다.

In [ ]:
# ============================================================
# 매매 거래에 전세 Lag 테이블 결합
# ============================================================

trade_rent = trade.copy()
before_merge_rows = len(trade_rent)

labels = [
    "전월",
    "과거3개월",
    "과거6개월",
]

gu_rent_cols = [
    col
    for label in labels
    for col in [
        f"자치구_{label}_전세평당가",
        f"자치구_{label}_전세거래량",
    ]
]

dong_rent_cols = [
    col
    for label in labels
    for col in [
        f"법정동_{label}_전세평당가",
        f"법정동_{label}_전세거래량",
        f"법정동_{label}_전세가율",
    ]
]

apt_rent_cols = [
    col
    for label in labels
    for col in [
        f"단지_{label}_전세평당가",
        f"단지_{label}_전세거래량",
    ]
]

gu_for_merge = (
    gu_market_features[
        [
            "sggCd",
            "거래월일자",
        ]
        + gu_rent_cols
    ]
    .rename(
        columns={
            "sggCd": "법정동시군구코드",
        }
    )
)

dong_for_merge = (
    dong_market_features[
        [
            "sggCd",
            "umdNm",
            "거래월일자",
        ]
        + dong_rent_cols
    ]
    .rename(
        columns={
            "sggCd": "법정동시군구코드",
            "umdNm": "법정동",
        }
    )
)

apt_for_merge = (
    apt_market_features[
        [
            "aptSeq",
            "sggCd",
            "umdNm",
            "거래월일자",
        ]
        + apt_rent_cols
    ]
    .rename(
        columns={
            "aptSeq": "단지일련번호",
            "sggCd": "법정동시군구코드",
            "umdNm": "법정동",
        }
    )
)

trade_rent = trade_rent.merge(
    gu_for_merge,
    on=[
        "법정동시군구코드",
        "거래월일자",
    ],
    how="left",
    validate="many_to_one",
)

trade_rent = trade_rent.merge(
    dong_for_merge,
    on=[
        "법정동시군구코드",
        "법정동",
        "거래월일자",
    ],
    how="left",
    validate="many_to_one",
)

trade_rent = trade_rent.merge(
    apt_for_merge,
    on=[
        "단지일련번호",
        "법정동시군구코드",
        "법정동",
        "거래월일자",
    ],
    how="left",
    validate="many_to_one",
)

if len(trade_rent) != before_merge_rows:
    raise AssertionError(
        "전월세 피처 결합 중 매매 행 수가 변경되었습니다."
    )

print(
    "결합 전/후:",
    before_merge_rows,
    "→",
    len(trade_rent),
)


In [ ]:
# ============================================================
# 단지 → 법정동 → 자치구 fallback
# ============================================================

for label, min_trades in [
    ("전월", 1),
    ("과거3개월", APT_MIN_TRADES),
    ("과거6개월", APT_MIN_TRADES),
]:
    apt_volume = trade_rent[
        f"단지_{label}_전세거래량"
    ]

    apt_price = trade_rent[
        f"단지_{label}_전세평당가"
    ].where(
        apt_volume >= min_trades
    )

    dong_price = trade_rent[
        f"법정동_{label}_전세평당가"
    ]

    gu_price = trade_rent[
        f"자치구_{label}_전세평당가"
    ]

    trade_rent[
        f"전세평당가_{label}"
    ] = (
        apt_price
        .combine_first(dong_price)
        .combine_first(gu_price)
    )

    trade_rent[
        f"전세거래량_{label}"
    ] = (
        apt_volume
        .where(apt_price.notna())
        .combine_first(
            trade_rent[
                f"법정동_{label}_전세거래량"
            ]
        )
        .combine_first(
            trade_rent[
                f"자치구_{label}_전세거래량"
            ]
        )
    )

    trade_rent[
        f"전세피처출처_{label}"
    ] = np.select(
        [
            apt_price.notna(),
            apt_price.isna()
            & dong_price.notna(),
            apt_price.isna()
            & dong_price.isna()
            & gu_price.notna(),
        ],
        [
            "단지",
            "법정동",
            "자치구",
        ],
        default="결측",
    )

display(
    trade_rent[
        [
            "전세피처출처_전월",
            "전세피처출처_과거3개월",
            "전세피처출처_과거6개월",
        ]
    ]
    .apply(pd.Series.value_counts)
    .fillna(0)
)


## 7. Leakage 검증

법정동 Lag 표본을 직접 재계산해
모든 원천 월이 대상 매매월보다 이전인지 확인한다.

In [ ]:
# ============================================================
# Lag 시계열 누수 검증
# ============================================================

def values_match(actual, expected):
    if pd.isna(actual) and pd.isna(expected):
        return True

    if pd.isna(actual) or pd.isna(expected):
        return False

    return np.isclose(
        float(actual),
        float(expected),
        rtol=1e-10,
        atol=1e-10,
    )


audit_candidates = (
    dong_market_features
    .dropna(
        subset=[
            "법정동_과거6개월_전세평당가"
        ]
    )
    .head(20)
)

audit_records = []

for _, row in audit_candidates.iterrows():
    target_month = row["거래월일자"]

    group_mask = (
        dong_panel["sggCd"].eq(
            row["sggCd"]
        )
        & dong_panel["umdNm"].eq(
            row["umdNm"]
        )
    )

    for window in ROLLING_WINDOWS:
        start_month = (
            target_month
            - pd.DateOffset(months=window)
        )

        source_rows = dong_panel.loc[
            group_mask
            & dong_panel["거래월일자"].ge(
                start_month
            )
            & dong_panel["거래월일자"].lt(
                target_month
            )
        ]

        valid_prices = (
            source_rows[
                "월중위전세평당가"
            ]
            .dropna()
        )

        expected_price = (
            valid_prices.median()
            if len(valid_prices)
            >= PRICE_MIN_MONTHS[window]
            else np.nan
        )

        expected_volume = (
            source_rows[
                "월전세거래량"
            ].sum()
            if len(source_rows) == window
            else np.nan
        )

        audit_records.append(
            {
                "sggCd": row["sggCd"],
                "umdNm": row["umdNm"],
                "대상월": target_month,
                "window": window,
                "가격일치": values_match(
                    row[
                        f"법정동_과거{window}개월_전세평당가"
                    ],
                    expected_price,
                ),
                "거래량일치": values_match(
                    row[
                        f"법정동_과거{window}개월_전세거래량"
                    ],
                    expected_volume,
                ),
                "원천월_대상월이전": (
                    source_rows.empty
                    or source_rows[
                        "거래월일자"
                    ].max()
                    < target_month
                ),
            }
        )

lag_audit = pd.DataFrame(
    audit_records
)

if not lag_audit.empty:
    assert lag_audit[
        [
            "가격일치",
            "거래량일치",
            "원천월_대상월이전",
        ]
    ].all().all()

print(
    "Lag 표본 검증:",
    len(lag_audit),
    "건 통과",
)


## 8. Feature-only 저장

V1 전월세 피처는 중복도가 높은 자치구·법정동·단지 원본 Lag를 모두 저장하지 않고,
fallback이 끝난 최종 피처와 법정동 전세가율만 남긴다.

최종 컬럼:
- 전세 평당가 Lag 1/3/6개월
- 전세 거래량 Lag 1/3/6개월
- 법정동 전세가율 Lag 1/3/6개월
- 전세 피처 출처 1/3/6개월

In [ ]:
# ============================================================
# 최종 전월세 feature-only 데이터 구성
# ============================================================

rent_feature_cols = [
    "전세평당가_전월",
    "전세평당가_과거3개월",
    "전세평당가_과거6개월",

    "전세거래량_전월",
    "전세거래량_과거3개월",
    "전세거래량_과거6개월",

    "법정동_전월_전세가율",
    "법정동_과거3개월_전세가율",
    "법정동_과거6개월_전세가율",

    "전세피처출처_전월",
    "전세피처출처_과거3개월",
    "전세피처출처_과거6개월",
]

df_fe_rent = trade_rent[
    ["transaction_id"]
    + rent_feature_cols
].copy()

target_derived_blocklist = {
    "거래금액",
    "log_price",
    "평당가",
    "log_평당가",
}

if target_derived_blocklist.intersection(
    df_fe_rent.columns
):
    raise AssertionError(
        "현재 거래가격 기반 컬럼이 "
        "전월세 피처에 포함되었습니다."
    )

print("shape:", df_fe_rent.shape)
print(
    "transaction_id 중복:",
    df_fe_rent[
        "transaction_id"
    ].duplicated().sum(),
)

display(
    df_fe_rent[
        rent_feature_cols
    ]
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
    .to_frame("결측비율")
)


In [ ]:
# # ============================================================
# # parquet 저장
# # ============================================================

# PROCESSED_DIR.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# df_fe_rent.to_parquet(
#     RENT_FEATURE_PATH,
#     index=False,
# )

# print(
#     "저장 완료:",
#     RENT_FEATURE_PATH,
# )

# print(
#     "최종 크기:",
#     df_fe_rent.shape,
# )
